In [130]:
%pip install pandas scikit-learn nltk
%pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [131]:
import pandas as pd
import re
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

### Load Dataset

In [132]:
file_path = "../data/raw/all_emotions-2.csv"

df = pd.read_csv(file_path)

print("shape:", df.shape)
df.head()

shape: (9021, 3)


,speaker,utterance,emotion
0,patient,I can feel the cold water on my teeth.,neutral
1,dentist,It's important to keep the area clean.,neutral
2,patient,I try to brush twice a day like you said.,neutral
3,dentist,You might feel a bit of pressure.,neutral
4,patient,Should I open wider?,neutral


In [133]:
print(df.columns.tolist())

['speaker', 'utterance', 'emotion']


In [134]:
print(df["speaker"].unique())
print(df["speaker"].value_counts())

['patient' 'dentist']
speaker
patient    4516
dentist    4505
Name: count, dtype: int64


### Data Filtering

In [135]:
df["speaker"] = df["speaker"].astype(str).str.lower().str.strip()

df = df[df["speaker"] == "patient"]

print("After filtering only patient data:", df.shape)
df.head()

After filtering only patient data: (4516, 3)


,speaker,utterance,emotion
0,patient,I can feel the cold water on my teeth.,neutral
2,patient,I try to brush twice a day like you said.,neutral
4,patient,Should I open wider?,neutral
6,patient,I am not paying for a mistake you made.,angry
8,patient,I can feel the cold water on my teeth.,neutral


### Data Cleaning

In [136]:
df = df[["utterance", "emotion"]].copy()
df.head()

,utterance,emotion
0,I can feel the cold water on my teeth.,neutral
2,I try to brush twice a day like you said.,neutral
4,Should I open wider?,neutral
6,I am not paying for a mistake you made.,angry
8,I can feel the cold water on my teeth.,neutral


In [137]:
df.dropna(subset=["utterance", "emotion"], inplace=True)

df["utterance"] = df["utterance"].astype(str)
df["emotion"] = df["emotion"].astype(str)

df = df[df["utterance"].str.strip() != ""]
df = df[df["emotion"].str.strip() != ""]

print("Shape after removing missing and empty values:", df.shape)

Shape after removing missing and empty values: (4516, 2)


### Text Cleaning

In [138]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["utterance"].apply(clean_text)

df = df[df["clean_text"].str.strip() != ""]

df[["utterance", "clean_text", "emotion"]].head()

,utterance,clean_text,emotion
0,I can feel the cold water on my teeth.,i can feel the cold water on my teeth,neutral
2,I try to brush twice a day like you said.,i try to brush twice a day like you said,neutral
4,Should I open wider?,should i open wider,neutral
6,I am not paying for a mistake you made.,i am not paying for a mistake you made,angry
8,I can feel the cold water on my teeth.,i can feel the cold water on my teeth,neutral


### Tokenization

In [139]:
def tokenize(text):
    return text.split()

df["tokens"] = df["clean_text"].apply(tokenize)

df[["clean_text", "tokens"]].head()

,clean_text,tokens
0,i can feel the cold water on my teeth,"[i, can, feel, the, cold, water, on, my, teeth]"
2,i try to brush twice a day like you said,"[i, try, to, brush, twice, a, day, like, you, ..."
4,should i open wider,"[should, i, open, wider]"
6,i am not paying for a mistake you made,"[i, am, not, paying, for, a, mistake, you, made]"
8,i can feel the cold water on my teeth,"[i, can, feel, the, cold, water, on, my, teeth]"


### Data Preparation

In [140]:
X_text = df["clean_text"]
y = df["emotion"]

print("Number of samples:", len(X_text))
print("\nLabel distribution:")
print(y.value_counts())

Number of samples: 4516

Label distribution:
emotion
neutral    2856
disgust     427
fear        395
angry       326
sad         278
happy       234
Name: count, dtype: int64


### TF-IDF Vectorization

In [141]:
vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    min_df=1
)

X_tfidf = vectorizer.fit_transform(X_text)

print("TF-IDF shape:", X_tfidf.shape)
print("\nตัวอย่าง feature 20 ตัวแรก:")
print(vectorizer.get_feature_names_out()[:20])

TF-IDF shape: (4516, 3000)

ตัวอย่าง feature 20 ตัวแรก:
['able' 'able to' 'about' 'about all' 'about checking' 'about cleaning'
 'about drilling' 'about having' 'about how' 'about it' 'about letting'
 'about my' 'about numbing' 'about pain' 'about pulling' 'about root'
 'about stitching' 'about that' 'about the' 'about this']


### TF-IDF Representation

In [142]:
tfidf_sample = pd.DataFrame(
    X_tfidf[:5].toarray(),
    columns=vectorizer.get_feature_names_out()
)

tfidf_sample.head()

,able,able to,about,about all,about checking,about cleaning,about drilling,about having,about how,about it,...,you start,you tell,you think,you told,you were,you would,youll,your,your receptionist,youre
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [143]:
import os

os.makedirs("../data/processed", exist_ok=True)

df.to_csv("../data/processed/cleaned_emotions_patient_ml.csv",
          index=False,
          encoding="utf-8-sig")

tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=vectorizer.get_feature_names_out()
)

tfidf_df["emotion"] = y.values

tfidf_df.to_csv("../data/processed/tfidf_features_patient_ml.csv",
                index=False,
                encoding="utf-8-sig")

print("Files saved successfully:")
print("- cleaned_emotions_patient_ml.csv")
print("- tfidf_features_patient_ml.csv")

Files saved successfully:
- cleaned_emotions_patient_ml.csv
- tfidf_features_patient_ml.csv


### Train / Test Split

In [144]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("Before SMOTE")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nTrain label distribution:")
print(y_train.value_counts())

print("\nTest label distribution:")
print(y_test.value_counts())

Before SMOTE
X_train shape: (3612, 3000)
X_test shape: (904, 3000)

Train label distribution:
emotion
neutral    2284
disgust     342
fear        316
angry       261
sad         222
happy       187
Name: count, dtype: int64

Test label distribution:
emotion
neutral    572
disgust     85
fear        79
angry       65
sad         56
happy       47
Name: count, dtype: int64


### Convert sparse matrix to dense

In [145]:
X_train_patient = X_train.toarray()
X_test_patient = X_test.toarray()

### Apply SMOTE on training set only

In [146]:
smote = SMOTE(random_state=42)

X_train_patient_sm, y_train_patient_sm = smote.fit_resample(
    X_train_patient, y_train
)

print("\nAfter SMOTE")
print("X_train_patient_sm shape:", X_train_patient_sm.shape)

print("\nTrain label distribution after SMOTE:")
print(pd.Series(y_train_patient_sm).value_counts())



After SMOTE
X_train_patient_sm shape: (13704, 3000)

Train label distribution after SMOTE:
emotion
neutral    2284
happy      2284
sad        2284
disgust    2284
angry      2284
fear       2284
Name: count, dtype: int64


###  Final ready data

In [147]:
X_train_final = X_train_patient_sm
y_train_final = y_train_patient_sm
X_test_final = X_test_patient
y_test_final = y_test

print("\nFinal data ready for model")
print("X_train_final:", X_train_final.shape)
print("y_train_final:", y_train_final.shape)
print("X_test_final:", X_test_final.shape)
print("y_test_final:", y_test_final.shape)



Final data ready for model
X_train_final: (13704, 3000)
y_train_final: (13704,)
X_test_final: (904, 3000)
y_test_final: (904,)


### Save train/test files

In [148]:
import os

os.makedirs("../data/processed", exist_ok=True)

feature_names = vectorizer.get_feature_names_out()

train_smote_df = pd.DataFrame(X_train_final, columns=feature_names)
train_smote_df["emotion"] = y_train_final.values

test_df = pd.DataFrame(X_test_final, columns=feature_names)
test_df["emotion"] = y_test_final.values

train_smote_df.to_csv(
    "../data/processed/patient_train_tfidf_smote_ml.csv",
    index=False,
    encoding="utf-8-sig"
)

test_df.to_csv(
    "../data/processed/patient_test_tfidf_ml.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nFiles saved successfully:")
print("- patient_train_tfidf_smote_ml.csv")
print("- patient_test_tfidf_ml.csv")


Files saved successfully:
- patient_train_tfidf_smote_ml.csv
- patient_test_tfidf_ml.csv


### Import Model

In [149]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import pandas as pd

In [150]:
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    # Train model
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    cm = confusion_matrix(y_test, y_pred)
    
    print(f"\n===== {model_name} =====")
    print(f"Accuracy      : {acc:.4f}")
    print(f"F1-score Macro: {f1_macro:.4f}")
    print(f"F1-score Weighted: {f1_weighted:.4f}")
    print("\nConfusion Matrix:")
    print(cm)
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    return {
        "Model": model_name,
        "Accuracy": acc,
        "F1_Macro": f1_macro,
        "F1_Weighted": f1_weighted,
        "Confusion_Matrix": cm
    }

### Logistic Regression

In [151]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)

lr_result = evaluate_model(
    lr_model,
    X_train_final,
    y_train_final,
    X_test_final,
    y_test_final,
    "Logistic Regression"
)


===== Logistic Regression =====
Accuracy      : 0.9170
F1-score Macro: 0.9149
F1-score Weighted: 0.9201

Confusion Matrix:
[[ 65   0   0   0   0   0]
 [  0  76   2   0   7   0]
 [  0   2  64   0  13   0]
 [  0   0   0  47   0   0]
 [  1  14  35   0 522   0]
 [  0   0   0   0   1  55]]

Classification Report:
              precision    recall  f1-score   support

       angry       0.98      1.00      0.99        65
     disgust       0.83      0.89      0.86        85
        fear       0.63      0.81      0.71        79
       happy       1.00      1.00      1.00        47
     neutral       0.96      0.91      0.94       572
         sad       1.00      0.98      0.99        56

    accuracy                           0.92       904
   macro avg       0.90      0.93      0.91       904
weighted avg       0.93      0.92      0.92       904



### Support Vector Machine

In [152]:
svm_model = SVC(kernel='linear', random_state=42)

svm_result = evaluate_model(
    svm_model,
    X_train_final,
    y_train_final,
    X_test_final,
    y_test_final,
    "Support Vector Machine"
)


===== Support Vector Machine =====
Accuracy      : 0.9137
F1-score Macro: 0.9114
F1-score Weighted: 0.9171

Confusion Matrix:
[[ 65   0   0   0   0   0]
 [  0  73   1   0  11   0]
 [  0   3  65   0  11   0]
 [  0   0   0  47   0   0]
 [  0   9  40   0 521   2]
 [  0   0   0   0   1  55]]

Classification Report:
              precision    recall  f1-score   support

       angry       1.00      1.00      1.00        65
     disgust       0.86      0.86      0.86        85
        fear       0.61      0.82      0.70        79
       happy       1.00      1.00      1.00        47
     neutral       0.96      0.91      0.93       572
         sad       0.96      0.98      0.97        56

    accuracy                           0.91       904
   macro avg       0.90      0.93      0.91       904
weighted avg       0.92      0.91      0.92       904



### Naive Bayes

In [153]:
nb_model = MultinomialNB()

nb_result = evaluate_model(
    nb_model,
    X_train_final,
    y_train_final,
    X_test_final,
    y_test_final,
    "Naive Bayes"
)


===== Naive Bayes =====
Accuracy      : 0.8595
F1-score Macro: 0.8397
F1-score Weighted: 0.8625

Confusion Matrix:
[[ 65   0   0   0   0   0]
 [  0  76   1   0   8   0]
 [  1   8  53   0  17   0]
 [  0   0   0  47   0   0]
 [  8  37  26  11 485   5]
 [  0   0   0   0   5  51]]

Classification Report:
              precision    recall  f1-score   support

       angry       0.88      1.00      0.94        65
     disgust       0.63      0.89      0.74        85
        fear       0.66      0.67      0.67        79
       happy       0.81      1.00      0.90        47
     neutral       0.94      0.85      0.89       572
         sad       0.91      0.91      0.91        56

    accuracy                           0.86       904
   macro avg       0.81      0.89      0.84       904
weighted avg       0.87      0.86      0.86       904



### Random Forest

In [154]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

rf_result = evaluate_model(
    rf_model,
    X_train_final,
    y_train_final,
    X_test_final,
    y_test_final,
    "Random Forest"
)


===== Random Forest =====
Accuracy      : 0.9237
F1-score Macro: 0.9078
F1-score Weighted: 0.9193

Confusion Matrix:
[[ 65   0   0   0   0   0]
 [  0  63   0   0  22   0]
 [  0   0  47   0  32   0]
 [  0   0   0  47   0   0]
 [  1   7   5   0 559   0]
 [  0   0   0   0   2  54]]

Classification Report:
              precision    recall  f1-score   support

       angry       0.98      1.00      0.99        65
     disgust       0.90      0.74      0.81        85
        fear       0.90      0.59      0.72        79
       happy       1.00      1.00      1.00        47
     neutral       0.91      0.98      0.94       572
         sad       1.00      0.96      0.98        56

    accuracy                           0.92       904
   macro avg       0.95      0.88      0.91       904
weighted avg       0.92      0.92      0.92       904



### Result

In [155]:
results_df = pd.DataFrame([
    {
        "Model": lr_result["Model"],
        "Accuracy": lr_result["Accuracy"],
        "F1_Macro": lr_result["F1_Macro"],
        "F1_Weighted": lr_result["F1_Weighted"]
    },
    {
        "Model": svm_result["Model"],
        "Accuracy": svm_result["Accuracy"],
        "F1_Macro": svm_result["F1_Macro"],
        "F1_Weighted": svm_result["F1_Weighted"]
    },
    {
        "Model": nb_result["Model"],
        "Accuracy": nb_result["Accuracy"],
        "F1_Macro": nb_result["F1_Macro"],
        "F1_Weighted": nb_result["F1_Weighted"]
    },
    {
        "Model": rf_result["Model"],
        "Accuracy": rf_result["Accuracy"],
        "F1_Macro": rf_result["F1_Macro"],
        "F1_Weighted": rf_result["F1_Weighted"]
    }
])

results_df = results_df.sort_values(by="F1_Macro", ascending=False)
results_df

,Model,Accuracy,F1_Macro,F1_Weighted
0,Logistic Regression,0.917035,0.914925,0.920075
1,Support Vector Machine,0.913717,0.911445,0.917144
3,Random Forest,0.923673,0.907753,0.919269
2,Naive Bayes,0.859513,0.839683,0.862483
